# 02 Plot Histones

# 02.1 Initialization

In [2]:
docker_run() {
    docker run --rm -i \
        -u $(id -u):$(id -g) \
        -v /home/dalbao:/home/dalbao \
        -v /etc/timezone:/etc/timezone:ro \
        -v /etc/localtime:/etc/localtime:ro \
        -w $(pwd) \
        "$@"
}

# Define software to use:
## deeptools for analysis and visualization of deep-sequencing data
deeptools() {
    docker_run quay.io/biocontainers/deeptools:3.5.6--pyhdfd78af_0 "$@"
}
deeptools plotHeatmap --version

# Define software to use:
## bedtools for bed file manipulation
bedtools() {
    docker_run staphb/bedtools:2.31.1 bedtools "$@"
}
bedtools --version

cd /home/dalbao/AlbaoRunx3Manuscript/cutnrun
mkdir -p 02_histones_Genes/background_corrected

mkdir -p failed for path /.config/matplotlib: [Errno 13] Permission denied: '/.config'
Matplotlib created a temporary cache directory at /tmp/matplotlib-7mkbs7s_ because there was an issue with the default path (/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
plotHeatmap 3.5.6
bedtools v2.31.1


In [3]:
# Define input bigWig files
basePath="/home/dalbao/AlbaoRunx3Manuscript/cutnrun/source_data/260713_reBAM2_noDeDup_CPM/04_reporting/igv/"

declare -A inputBigwig

for target in H3K4me3 H3K36me3 IgG; do
    for group in shCd19 shRunx3 memory early late terminal; do
        name="${group}_${target}"
        inputBigwig[$name]="${basePath}${group}_${target}_R1.bigWig"
        if [ ! -f "${inputBigwig[$name]}" ]; then
            echo "Error: Input file ${inputBigwig[$name]} does not exist."
            exit 1
        fi
        echo ${inputBigwig[$name]}
    done
done

/home/dalbao/AlbaoRunx3Manuscript/cutnrun/source_data/260713_reBAM2_noDeDup_CPM/04_reporting/igv/shCd19_H3K4me3_R1.bigWig
/home/dalbao/AlbaoRunx3Manuscript/cutnrun/source_data/260713_reBAM2_noDeDup_CPM/04_reporting/igv/shRunx3_H3K4me3_R1.bigWig
/home/dalbao/AlbaoRunx3Manuscript/cutnrun/source_data/260713_reBAM2_noDeDup_CPM/04_reporting/igv/memory_H3K4me3_R1.bigWig
/home/dalbao/AlbaoRunx3Manuscript/cutnrun/source_data/260713_reBAM2_noDeDup_CPM/04_reporting/igv/early_H3K4me3_R1.bigWig
/home/dalbao/AlbaoRunx3Manuscript/cutnrun/source_data/260713_reBAM2_noDeDup_CPM/04_reporting/igv/late_H3K4me3_R1.bigWig
/home/dalbao/AlbaoRunx3Manuscript/cutnrun/source_data/260713_reBAM2_noDeDup_CPM/04_reporting/igv/terminal_H3K4me3_R1.bigWig
/home/dalbao/AlbaoRunx3Manuscript/cutnrun/source_data/260713_reBAM2_noDeDup_CPM/04_reporting/igv/shCd19_H3K36me3_R1.bigWig
/home/dalbao/AlbaoRunx3Manuscript/cutnrun/source_data/260713_reBAM2_noDeDup_CPM/04_reporting/igv/shRunx3_H3K36me3_R1.bigWig
/home/dalbao/AlbaoRun

In [4]:
declare -A correctedSubtract
declare -A correctedLog2

for group in memory early late terminal shCd19 shRunx3; do
    for signal in H3K4me3 H3K36me3; do
        name="${group}_${signal}"
        subtractOut="02_histones_Genes/background_corrected/${name}_subtract.bigWig"
        log2Out="02_histones_Genes/background_corrected/${name}_log2.bigWig"

        deeptools bigwigCompare \
            -b1 "${inputBigwig[$name]}" \
            -b2 "${inputBigwig["${group}_IgG"]}" \
            --operation subtract \
            --numberOfProcessors 36 \
            -o "$subtractOut"

        deeptools bigwigCompare \
            -b1 "${inputBigwig[$name]}" \
            -b2 "${inputBigwig["${group}_IgG"]}" \
            --operation log2 \
            --numberOfProcessors 36 \
            -o "$log2Out"

        correctedSubtract[$name]="$subtractOut"
        correctedLog2[$name]="$log2Out"
    done
done

mkdir -p failed for path /.config/matplotlib: [Errno 13] Permission denied: '/.config'
Matplotlib created a temporary cache directory at /tmp/matplotlib-hyy838ef because there was an issue with the default path (/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
mkdir -p failed for path /.config/matplotlib: [Errno 13] Permission denied: '/.config'
Matplotlib created a temporary cache directory at /tmp/matplotlib-2a6tniyb because there was an issue with the default path (/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
mkdir -p failed for path /.config/matplotlib: [Errno 13] Permission denied: '/.config'
Matplotlib created a temporary cache directory at /tmp/matplotlib-a2au6c2m 

In [6]:
for signal in H3K4me3 H3K36me3; do
    deeptools computeMatrix scale-regions \
        -S \
        02_histones_Genes/background_corrected/shCd19_${signal}_log2.bigWig \
        02_histones_Genes/background_corrected/shRunx3_${signal}_log2.bigWig \
        02_histones_Genes/background_corrected/memory_${signal}_log2.bigWig \
        02_histones_Genes/background_corrected/early_${signal}_log2.bigWig \
        02_histones_Genes/background_corrected/late_${signal}_log2.bigWig \
        02_histones_Genes/background_corrected/terminal_${signal}_log2.bigWig \
        -R \
        02_histones_Genes/sets/Runx3OE_Up.bed \
        02_histones_Genes/sets/Runx3OE_Down.bed \
        02_histones_Genes/sets/shRunx3_Up.bed \
        02_histones_Genes/sets/shRunx3_Down.bed \
        02_histones_Genes/sets/Exh_Up.bed \
        02_histones_Genes/sets/Exh_Down.bed \
        02_histones_Genes/sets/TcmE_Up.bed \
        02_histones_Genes/sets/TcmE_Down.bed \
        --beforeRegionStartLength 3000 \
        --regionBodyLength 5000 \
        --afterRegionStartLength 3000 \
        --skipZeros \
        --numberOfProcessors 36 \
        -out "02_histones_Genes/${signal}.gz"
done

mkdir -p failed for path /.config/matplotlib: [Errno 13] Permission denied: '/.config'
Matplotlib created a temporary cache directory at /tmp/matplotlib-9z98ym9o because there was an issue with the default path (/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Skipping Rrm2, due to being absent in the computeMatrix output.
Skipping Rrm2_r1, due to being absent in the computeMatrix output.
Skipping Rrm2_r2, due to being absent in the computeMatrix output.
Skipping Rrm2_r3, due to being absent in the computeMatrix output.
Skipping Csf2ra, due to being absent in the computeMatrix output.
Skipping Ctla4_r3, due to being absent in the computeMatrix output.
Skipping Ctla4_r4, due to being absent in the computeMatrix output.
Skipping Gvin1, due to being absent in the computeMatrix output.
Skipping Gvin1_r1, due to being absent in the com

In [7]:
deeptools plotHeatmap \
    -m "02_histones_Genes/H3K4me3.gz" \
    -out "02_histones_Genes/H3K4me3.pdf" \
    --sortUsing sum \
    --colorMap "RdYlBu_r" \
    --regionsLabel "RUNX3OE up" "RUNX3OE down" "shRunx3 up" "shRunx3 down" \
                    "Exh_Up" "Exh_Down" "TcmE_Up" "TcmE_Down" \
    --samplesLabel "shCd19" "shRunx3" \
                   "memory" "early" "late" "terminal"


mkdir -p failed for path /.config/matplotlib: [Errno 13] Permission denied: '/.config'
Matplotlib created a temporary cache directory at /tmp/matplotlib-6gu51x1b because there was an issue with the default path (/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [8]:
deeptools plotHeatmap \
    -m "02_histones_Genes/H3K36me3.gz" \
    -out "02_histones_Genes/H3K36me3.pdf" \
    --sortUsing sum \
    --colorMap "RdYlBu_r" \
    --regionsLabel "RUNX3OE up" "RUNX3OE down" "shRunx3 up" "shRunx3 down" \
                    "Exh_Up" "Exh_Down" "TcmE_Up" "TcmE_Down" \
    --samplesLabel "shCd19" "shRunx3" \
                   "memory" "early" "late" "terminal"

mkdir -p failed for path /.config/matplotlib: [Errno 13] Permission denied: '/.config'
Matplotlib created a temporary cache directory at /tmp/matplotlib-yn4_mh9_ because there was an issue with the default path (/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [9]:
for signal in H3K4me3 H3K36me3; do
    deeptools computeMatrix scale-regions \
        -S \
        02_histones_Genes/background_corrected/memory_${signal}_log2.bigWig \
        02_histones_Genes/background_corrected/early_${signal}_log2.bigWig \
        02_histones_Genes/background_corrected/late_${signal}_log2.bigWig \
        02_histones_Genes/background_corrected/terminal_${signal}_log2.bigWig \
        -R \
        02_histones_Genes/sets/Runx3OE_Up.bed \
        02_histones_Genes/sets/Runx3OE_Down.bed \
        02_histones_Genes/sets/shRunx3_Up.bed \
        02_histones_Genes/sets/shRunx3_Down.bed \
        --beforeRegionStartLength 3000 \
        --regionBodyLength 5000 \
        --afterRegionStartLength 3000 \
        --skipZeros \
        --numberOfProcessors 36 \
        -out "02_histones_Genes/${signal}.gz"
done

mkdir -p failed for path /.config/matplotlib: [Errno 13] Permission denied: '/.config'
Matplotlib created a temporary cache directory at /tmp/matplotlib-5mk9jbyj because there was an issue with the default path (/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Skipping Rrm2, due to being absent in the computeMatrix output.
Skipping Rrm2_r1, due to being absent in the computeMatrix output.
Skipping Rrm2_r2, due to being absent in the computeMatrix output.
Skipping Rrm2_r3, due to being absent in the computeMatrix output.
mkdir -p failed for path /.config/matplotlib: [Errno 13] Permission denied: '/.config'
Matplotlib created a temporary cache directory at /tmp/matplotlib-gx5g8stt because there was an issue with the default path (/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writabl

In [17]:
deeptools plotHeatmap \
    -m "02_histones_Genes/H3K4me3.gz" \
    -out "02_histones_Genes/H3K4me3.pdf" \
    --sortUsing sum \
    --colorMap "RdYlBu_r" \
    --regionsLabel "RUNX3OE up" "RUNX3OE down" "shRunx3 up" "shRunx3 down" \
    --samplesLabel "memory" "early" "late" "terminal" --zMin 0

deeptools plotHeatmap \
    -m "02_histones_Genes/H3K36me3.gz" \
    -out "02_histones_Genes/H3K36me3.pdf" \
    --sortUsing sum \
    --colorMap "RdYlBu_r" \
    --regionsLabel "RUNX3OE up" "RUNX3OE down" "shRunx3 up" "shRunx3 down" \
    --samplesLabel "memory" "early" "late" "terminal" --zMin 0


mkdir -p failed for path /.config/matplotlib: [Errno 13] Permission denied: '/.config'
Matplotlib created a temporary cache directory at /tmp/matplotlib-h28c_ez8 because there was an issue with the default path (/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
mkdir -p failed for path /.config/matplotlib: [Errno 13] Permission denied: '/.config'
Matplotlib created a temporary cache directory at /tmp/matplotlib-x2uzgvh5 because there was an issue with the default path (/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [12]:
for signal in H3K4me3 H3K36me3; do
    deeptools computeMatrix scale-regions \
        -S \
        02_histones_Genes/background_corrected/shCd19_${signal}_log2.bigWig \
        02_histones_Genes/background_corrected/shRunx3_${signal}_log2.bigWig \
        -R \
        02_histones_Genes/sets/Runx3OE_Up.bed \
        02_histones_Genes/sets/Runx3OE_Down.bed \
        02_histones_Genes/sets/shRunx3_Up.bed \
        02_histones_Genes/sets/shRunx3_Down.bed \
        --beforeRegionStartLength 3000 \
        --regionBodyLength 5000 \
        --afterRegionStartLength 3000 \
        --skipZeros \
        --numberOfProcessors 36 \
        -out "02_histones_Genes/${signal}_controls.gz"
done

mkdir -p failed for path /.config/matplotlib: [Errno 13] Permission denied: '/.config'
Matplotlib created a temporary cache directory at /tmp/matplotlib-knjn_tiv because there was an issue with the default path (/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Skipping Rrm2, due to being absent in the computeMatrix output.
Skipping Rrm2_r1, due to being absent in the computeMatrix output.
Skipping Rrm2_r2, due to being absent in the computeMatrix output.
Skipping Rrm2_r3, due to being absent in the computeMatrix output.
mkdir -p failed for path /.config/matplotlib: [Errno 13] Permission denied: '/.config'
Matplotlib created a temporary cache directory at /tmp/matplotlib-op4m9jr4 because there was an issue with the default path (/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writabl

In [16]:
deeptools plotHeatmap \
    -m "02_histones_Genes/H3K4me3_controls.gz" \
    -out "02_histones_Genes/H3K4me3_controls.pdf" \
    --sortUsing sum \
    --colorMap "RdYlBu_r" \
    --regionsLabel "RUNX3OE up" "RUNX3OE down" "shRunx3 up" "shRunx3 down" \
    --samplesLabel "shCd19" "shRunx3" --zMin 0

deeptools plotHeatmap \
    -m "02_histones_Genes/H3K36me3_controls.gz" \
    -out "02_histones_Genes/H3K36me3_controls.pdf" \
    --sortUsing sum \
    --colorMap "RdYlBu_r" \
    --regionsLabel "RUNX3OE up" "RUNX3OE down" "shRunx3 up" "shRunx3 down" \
    --samplesLabel "shCd19" "shRunx3" --zMin 0

mkdir -p failed for path /.config/matplotlib: [Errno 13] Permission denied: '/.config'
Matplotlib created a temporary cache directory at /tmp/matplotlib-igo9ixic because there was an issue with the default path (/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
mkdir -p failed for path /.config/matplotlib: [Errno 13] Permission denied: '/.config'
Matplotlib created a temporary cache directory at /tmp/matplotlib-ap5eoqg4 because there was an issue with the default path (/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
